# Estudo longitudinal de Apresentções

Constrói uma base de dados de apresentações artísticas do Siplan para análise longitudinal
(pré-pandemia 2018-19 / retomada 2022-23 / pós-pandemia 2024-25 / atual 2026).

## O que o script faz, célula a célula

| # | O que faz |
|---|---|
| **SQL** | Busca no Hive todas as ações com , aprovadas, não internas. Classifica  via RLIKE em +; agrega  por sessão filtrando . |
| **panorama_ano** | Conta ações e sessões únicas por período e ano. |
| **panorama_ling** | Mesmo recorte, decomposto por linguagem. |
| **precificação** | Extrai  (Pago/Gratuito),  e  do campo  via regex. |
| **período_dia** | Categoriza hora de  em Madrugada / Manhã / Tarde / Noite. |
| **tipo_local** | Classifica em *Na UO*, *Fora da UO* ou *Online*. |
| **local_grupo_apresentacoes** | Classifica o espaço em categorias (Teatros, Auditórios, Palco Externo, Ginásios, Galpão etc.) via lista de regras ordens (REGRAS_LOCAL_GRUPO). Para adicionar novas regras: inserir  na posição correta — específicas antes, gerais depois. |
| **unidades** | Cruza com  para trazer  e . |
| **export** | Salva . |

## Próximos passos planejados
- Análise estatística: evolução de sessões e público por tipo de espaço, UO, linguagem, período do ano e semana — com limpeza de outliers.
- Mesmo pipeline para **Cursos e Oficinas** e **Ações mediadas**.


## Acesso ao HIVE

In [20]:
import warnings
import numpy as np
import pandas as pd
import pyodbc
from pathlib import Path

OUTPUT_PATH = Path('output')
OUTPUT_PATH.mkdir(exist_ok=True)
DATA_INICIAL = '2026-01-01'

# Ajuste a string abaixo se o DSN exigir parÃ¢metros adicionais.
# Para autenticaÃ§Ã£o Kerberos, normalmente basta usar o DSN configurado.
KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Authentication=Kerberos;'
# Se o driver exigir Trusted Connection no Windows, use:
# KERBEROS_CONN_STR = 'DSN=DSN_HIVE_64;Trusted_Connection=Yes;'


def get_connection() -> pyodbc.Connection:
    return pyodbc.connect(KERBEROS_CONN_STR, autocommit=True)


def query_to_df(query: str) -> pd.DataFrame:
    with get_connection() as conn:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                'ignore',
                message='.*SQLAlchemy connectable.*',
                category=UserWarning,
                module='pandas.io.sql',
            )
            return pd.read_sql_query(query, conn)


In [31]:
sql_raw_acoes = f"""
SELECT
    atividade_id, uo, nome, complemento, linguagem,
    desc_subprograma, desc_modalidade, desc_realizacao, formato,
    areaprogramatica_nome, estimativa_publico, lugares, precificacao_desc,
    sessao_id, datainicio, datafinal, local_nome, local_grupo,
    dia_semana_sessao,
    SUM(presentes) AS presentes
FROM (
    SELECT
        acao.atividade_id,
        acao.uo,
        acao.nome,
        acao.complemento,
        CASE
            WHEN LOWER(CONCAT(COALESCE(acao.desc_modalidade, ''), ' ', COALESCE(acao.desc_subprograma, ''))) RLIKE 'circo' THEN 'Circo'
            WHEN LOWER(CONCAT(COALESCE(acao.desc_modalidade, ''), ' ', COALESCE(acao.desc_subprograma, ''))) RLIKE 'dança|danca' THEN 'Dança'
            WHEN LOWER(CONCAT(COALESCE(acao.desc_modalidade, ''), ' ', COALESCE(acao.desc_subprograma, ''))) RLIKE 'música|musica' THEN 'Música'
            WHEN LOWER(CONCAT(COALESCE(acao.desc_modalidade, ''), ' ', COALESCE(acao.desc_subprograma, ''))) RLIKE 'literatura' THEN 'Literatura'
            WHEN LOWER(CONCAT(COALESCE(acao.desc_modalidade, ''), ' ', COALESCE(acao.desc_subprograma, ''))) RLIKE 'audiovisual|cinema|vídeo' THEN 'Audiovisual'
            WHEN LOWER(CONCAT(COALESCE(acao.desc_modalidade, ''), ' ', COALESCE(acao.desc_subprograma, ''))) RLIKE 'teatro' THEN 'Teatro'
            WHEN acao.linguagem IN ('Teatro', 'Dança', 'Circo', 'Música', 'Literatura', 'Audiovisual') THEN acao.linguagem
            ELSE NULL
        END AS linguagem,
        acao.desc_subprograma,
        acao.desc_modalidade,
        acao.desc_realizacao,
        acao.linguagem as categoria,
        acao.formato,
        acao.areaprogramatica_nome,
        acao.precificacao_desc,
        dt.local_nome,
        dt.grupo_nome AS local_grupo,
        dt.sessao_id,
        dt.datainicio,
        dt.datafinal,
        SUBSTR(mens.dia_semana_sessao, 1, 3) AS dia_semana_sessao,
        acao.estimativa_publico,
        acao.lugares,
        mens.valor_mensurador AS presentes
    FROM stg_estatistico.siplan_acao acao
    INNER JOIN stg_estatistico.siplan_data dt
        ON acao.atividade_id = dt.atividade_id
    LEFT JOIN stg_estatistico.siplan_lancamento_mensurador mens
        ON dt.sessao_id = mens.sessao_id
       AND LOWER(mens.lancamento_status) = 'aprovado'
       AND mens.valor_mensurador IS NOT NULL
       AND mens.grupo_nome IN ('Público', 'Presenças')
    WHERE acao.status_atividade = 'APROVADO'
      AND (LOWER(acao.desc_realizacao) LIKE 'apresent%' or LOWER(acao.desc_realizacao) LIKE 'exibi%')
      AND LOWER(acao.desc_realizacao) not LIKE '%esportiva%'
      AND LOWER(acao.desc_subprograma) not LIKE '%esportivo%'
      AND acao.uso_interno = 0
      AND acao.manutencao = 0
      AND acao.regular = 0
) t
GROUP BY
    atividade_id, uo, nome, complemento, linguagem,
    desc_subprograma, desc_modalidade, desc_realizacao, formato,
    areaprogramatica_nome, estimativa_publico, lugares, precificacao_desc,
    sessao_id, datainicio, datafinal, local_nome, local_grupo,
    dia_semana_sessao
"""

In [ ]:
df_raw_acoes_preview = query_to_df(sql_raw_acoes)
# display(df_raw_acoes_preview)


## Panorama geral

In [ ]:
PERIODOS = {
    2018: "Pré-pandemia",
    2019: "Pré-pandemia",
    2022: "Retomada",
    2023: "Retomada",
    2024: "Pós-pandemia",
    2025: "Pós-pandemia",
    2026: "Atual",
}

df = df_raw_acoes_preview.copy()
df["ano"] = pd.to_datetime(df["datainicio"]).dt.year
df = df[df["ano"].isin(PERIODOS)].copy()
df["periodo"] = df["ano"].map(PERIODOS)

# --- Por ano ---
panorama_ano = (
    df.groupby(["periodo", "ano"], sort=False)
    .agg(
        acoes=("atividade_id", "nunique"),
        sessoes=("sessao_id", "nunique"),
    )
    .reset_index()
    .sort_values("ano")
    .reset_index(drop=True)
)
print("Panorama por ano")
# display(panorama_ano)

# --- Por linguagem ---
panorama_ling = (
    df.groupby(["periodo", "ano", "linguagem"], sort=False, dropna=False)
    .agg(
        acoes=("atividade_id", "nunique"),
        sessoes=("sessao_id", "nunique"),
    )
    .reset_index()
    .sort_values(["ano", "linguagem"])
    .reset_index(drop=True)
)
print("Panorama por linguagem")
# display(panorama_ling)

# --- Pivot: acoes por linguagem x ano ---
pivot_acoes = (
    panorama_ling
    .pivot_table(index="linguagem", columns="ano", values="acoes", aggfunc="sum", fill_value=0)
)
pivot_sessoes = (
    panorama_ling
    .pivot_table(index="linguagem", columns="ano", values="sessoes", aggfunc="sum", fill_value=0)
)

pivot_combinado = pd.concat(
    {"Ações": pivot_acoes, "Sessões": pivot_sessoes},
    axis=1,
)
print("Pivot: linguagem x ano (Ações / Sessões)")
display(pivot_combinado)


Panorama por ano


,periodo,ano,acoes,sessoes
0,Pré-pandemia,2018,12692,21590
1,Pré-pandemia,2019,12783,21838
2,Retomada,2022,8259,26139
3,Retomada,2023,10182,23738
4,Pós-pandemia,2024,10600,16896
5,Pós-pandemia,2025,11011,17049
6,Atual,2026,4549,7180


Panorama por linguagem


,periodo,ano,linguagem,acoes,sessoes
0,Pré-pandemia,2018,Audiovisual,2749,5143
1,Pré-pandemia,2018,Circo,967,1594
2,Pré-pandemia,2018,Dança,844,1439
3,Pré-pandemia,2018,Literatura,1342,2459
4,Pré-pandemia,2018,Música,4392,5659
5,Pré-pandemia,2018,Teatro,1915,4511
6,Pré-pandemia,2018,NaN,483,785
7,Pré-pandemia,2019,Audiovisual,2621,4741
8,Pré-pandemia,2019,Circo,942,1500
9,Pré-pandemia,2019,Dança,915,1424


Pivot: linguagem x ano (Ações / Sessões)


Ações                                     Sessões               \
ano          2018  2019  2022  2023  2024  2025  2026    2018  2019   2022   
linguagem                                                                    
Audiovisual  2749  2621  1936  2007  2061  2309   863    5143  4741  15300   
Circo         967   942   681   925   943  1081   461    1594  1500   1045   
Dança         844   915   502   661   592   678   271    1439  1424    803   
Literatura   1342  1259   733   876   903   825   267    2459  2413   1325   
Música       4392  4497  2739  3850  4192  4329  1950    5659  5625   3538   
Teatro       1915  1797  1380  1484  1516  1509   658    4511  4654   3625   

                                      
ano           2023  2024  2025  2026  
linguagem                             
Audiovisual  10404  2960  3326  1202  
Circo         1426  1564  1663   761  
Dança         1079   983  1104   440  
Literatura    1728  1751  1399   436  
Música        4799  5213  5271  2471  
Teatro        3702  3706  3770  1739

In [ ]:
import re

# Evita duplicar colunas a cada re-execucao
df_raw_acoes_preview = df_raw_acoes_preview.drop(
    columns=[c for c in ['acesso', 'maior_valor', 'menor_valor'] if c in df_raw_acoes_preview.columns]
)

def _parse_precificacao(desc):
    if pd.isna(desc) or str(desc).strip() in ('', 'None'):
        return None, None, None

    # Captura numeros decimais: 10.00 / .01 / 1.234,56 / 70,00
    nums = re.findall('\d*[.,]\d{2}', desc)
    valores = []
    for n in nums:
        try:
            if "," in n and "." in n:
                n = n.replace(".", "").replace(",", ".")
            else:
                n = n.replace(",", ".")
            v = float(n)
            if v < 1000:
                valores.append(v)
        except ValueError:
            pass

    has_gratis = bool(re.search('gr[aá]ti', desc, re.IGNORECASE))

    if not valores:
        return 'Gratuito', 0.0, 0.0

    if has_gratis:
        valores.append(0.0)

    return 'Pago', max(valores), min(valores)

parsed = df_raw_acoes_preview['precificacao_desc'].apply(
    lambda d: pd.Series(_parse_precificacao(d), index=['acesso', 'maior_valor', 'menor_valor'])
)
df_raw_acoes_preview = pd.concat([df_raw_acoes_preview, parsed], axis=1)

# with pd.option_context("display.max_rows", None):
#    display(
#        df_raw_acoes_preview[['precificacao_desc', 'acesso', 'maior_valor', 'menor_valor']]
#        .drop_duplicates('precificacao_desc')
#        .sort_values('precificacao_desc')
#        .reset_index(drop=True)
#    )


,precificacao_desc,acesso,maior_valor,menor_valor
0,Aberto ao público,Pago,NaN,NaN
1,Aberto ao público - *Vagas limitadas. Inscriç...,Pago,NaN,NaN
2,Aberto ao público - **Senhas limitadas devem s...,Pago,NaN,NaN
3,Aberto ao público - .,Pago,NaN,NaN
4,Aberto ao público - 6,Pago,NaN,NaN
...,...,...,...,...
302,Inscrição - R$ 70.00 / R$ 35.00 / R$ 21.00,Pago,NaN,NaN
303,Inscrição - R$ 8.00 / R$ 8.00 / Grátis,Pago,NaN,NaN
304,Inscrição - R$ 8.50 / R$ 8.50 / R$ 5.00,Pago,NaN,NaN
305,Inscrição - R$ 80.00 / R$ 40.00 / R$ 24.00,Pago,NaN,NaN


In [ ]:
def _periodo_dia(dt):
    if pd.isna(dt):
        return None
    h = pd.to_datetime(dt).hour
    if h < 5:    return 'Madrugada'
    if h < 12:   return 'Manhã'
    if h < 18:   return 'Tarde'
    return 'Noite'

df_raw_acoes_preview['periodo_dia'] = df_raw_acoes_preview['datainicio'].apply(_periodo_dia)


In [ ]:
# Cada regra e um dict com chaves opcionais: local, local_ou, grupo, uo, resultado.
# Todas as chaves presentes devem casar (AND). local_ou aceita lista de termos (OR).
# A ordem importa: primeira regra que casar vence.
REGRAS_LOCAL_GRUPO = [
    # --- Casos especificos antes das regras gerais ---
    dict(local='camarim',                             resultado='Outros'),
    dict(local='palco da comedoria',                  resultado='Comedorias'),
    dict(local='comedoria',             uo='68',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='comedoria',             uo='63',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='caixa preta',           uo='56',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='térreo',                uo='65',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='restaurante - térreo',  uo='64',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='espaço cênico',         uo='71',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='lanchonete',            uo='71',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='quadra 2',              uo='54',      resultado='Outros'),
    dict(local='ginásio',               uo='54',      resultado='Ginásios de eventos'),
    dict(local='ginásio',               uo='73',      resultado='Ginásios de eventos'),
    dict(local='ginásio',               uo='77',      resultado='Ginásios de eventos'),
    dict(local='ginásio',               uo='93',      resultado='Ginásios de eventos'),
    dict(local='ginásio',               uo='80',      resultado='Ginásios de eventos'),
    dict(local='ginásio',               uo='86',      resultado='Ginásios de eventos'),
    dict(local='ginásio',               uo='84',      resultado='Ginásios de eventos'),
    dict(local='eventos',               uo='83',      resultado='Ginásios de eventos'),
    dict(local='eventos',               uo='82',      resultado='Ginásios de eventos'),
    dict(grupo='espetáculo',            uo='68',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='cênico',                uo='63',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='palco externo',         uo='81',      resultado='Palco Externo'),
    dict(local='teatro [externo]',      uo='92',      resultado='Palco Externo'),
    dict(local='pau brasil',            uo='55',      resultado='Palco Externo'),
    dict(local='tenda',                 uo='72',      resultado='Palco Externo'),
    dict(local='quiosque b',            uo='79',      resultado='Palco Externo'),
    dict(grupo='eventos',               uo='56',      resultado='Palco Externo'),
    dict(grupo='auditório',             uo='89',      resultado='Outros'),
    dict(grupo='auditório',             uo='71',      resultado='Outros'),
    dict(local='teatro [interno]',      uo='92',      resultado='Teatros'),
    dict(local='galpão',                uo='63',      resultado='Salas de Espetáculo e Comedorias'),
    dict(local='solário',               uo='77',      resultado='Praças e Convivências'),
    dict(local='pça coberta',                         resultado='Praças e Convivências'),
    # --- Regras gerais (sem UO) ---
    dict(grupo='cinesesc',                            resultado='Cinesesc'),
    dict(grupo='auditório',                           resultado='Auditórios'),
    dict(local_ou=['convivência', 'praça'],           resultado='Praças e Convivências'),
    dict(grupo='arena',                               resultado='Praças e Convivências'),
    dict(local='arena',                               resultado='Praças e Convivências'),
    dict(grupo='teatro',                              resultado='Teatros'),
    dict(local='arte ii',                             resultado='Salas de Espetáculo e Comedorias'),
    dict(grupo='tenda',                               resultado='Praças e Convivências'),
    dict(grupo='praça de eventos',                    resultado='Praças e Convivências'),
    dict(grupo='múltiplo uso',                        resultado='Múltiplo Uso'),
    dict(grupo='oficina',                             resultado='Múltiplo Uso'),
    dict(grupo='tecnologias',                         resultado='Múltiplo Uso'),
    dict(grupo='praça de eventos',                    resultado='Praças e Convivências'),
    dict(grupo='quadra poliesportiva',                resultado='Quadras Poliestportivas'),
    dict(grupo='comedoria',                           resultado='Comedorias'),
    dict(grupo='galpão',                              resultado='Galpão'),
    dict(local='cpt',                                 resultado='Salas de Espetáculo e Comedorias'),
    dict(grupo='sala de espetáculo',                  resultado='Salas de Espetáculo e Comedorias'),
#    dict(local='cpt',                                 resultado='Outras Salas de Espetáculo'),
#    dict(grupo='sala de espetáculo',                  resultado='Outras Salas de Espetáculo'),
    dict(local='galpão',                              resultado='Galpão'),
]

def _classifica_local(local, grupo, uo):
    local_l = '' if pd.isna(local) else str(local).lower()
    grupo_l = '' if pd.isna(grupo) else str(grupo).lower()
    uo_l    = '' if pd.isna(uo)    else str(uo).lower()

    for r in REGRAS_LOCAL_GRUPO:
        if 'local'    in r and r['local'].lower()             not in local_l: continue
        if 'local_ou' in r and not any(p.lower() in local_l for p in r['local_ou']): continue
        if 'grupo'    in r and r['grupo'].lower()             not in grupo_l: continue
        if 'uo'       in r and r['uo']                        not in uo_l:    continue
        return r['resultado']

    return 'Outros'


def _tipo_local(local, grupo):
    local_l = '' if pd.isna(local) else str(local).lower()
    grupo_l = '' if pd.isna(grupo) else str(grupo).lower()
    if 'online' in local_l or 'online' in grupo_l:
        return 'Online'
    if 'fora da unidade' in grupo_l:
        return 'Fora da UO'
    return 'Na UO'


df_raw_acoes_preview['tipo_local'] = df_raw_acoes_preview.apply(
    lambda row: _tipo_local(row['local_nome'], row['local_grupo']), axis=1
)
df_raw_acoes_preview['local_grupo_apresentacoes'] = df_raw_acoes_preview.apply(
    lambda row: _classifica_local(row['local_nome'], row['local_grupo'], row['uo']), axis=1
)


In [ ]:
panorama_local = (
    df_raw_acoes_preview
    .groupby(['tipo_local', 'local_grupo_apresentacoes'], dropna=False)
    .agg(
        acoes=('atividade_id', 'nunique'),
        sessoes=('sessao_id', 'nunique'),
    )
    .sort_values(['tipo_local', 'sessoes'], ascending=[True, False])
    .reset_index()
)
# display(panorama_local)


,tipo_local,local_grupo_apresentacoes,acoes,sessoes
0,Fora da UO,Outros,6375,11041
1,Fora da UO,Praças e Convivências,400,535
2,Fora da UO,Outras Salas de Espetáculo,1,1
3,Na UO,Teatros,20713,33031
4,Na UO,Praças e Convivências,21417,28674
5,Na UO,Outros,9396,19607
6,Na UO,Salas de Espetáculo e Comedorias,5387,11887
7,Na UO,Cinesesc,4534,9467
8,Na UO,Auditórios,4580,9141
9,Na UO,Múltiplo Uso,3169,6617


In [ ]:
# Diagnostico: combinacoes ainda em Outros (mais impactantes primeiro)
# df_outros = (
#     df_raw_acoes_preview[df_raw_acoes_preview['local_grupo_apresentacoes'] == 'Outros']
#     .groupby(['uo', 'local_nome', 'local_grupo'], dropna=False)
#     .agg(
#         acoes=('atividade_id', 'nunique'),
#         sessoes=('sessao_id', 'nunique'),
#     )
#     .sort_values('sessoes', ascending=False)
#     .reset_index()
# )
# with pd.option_context('display.max_rows', None, 'display.max_colwidth', 60):
#     display(df_outros)


,uo,local_nome,local_grupo,acoes,sessoes
0,59,Ação Online,Ação Online,1814,38713
1,59,Sala de exibição,Sala Cinesesc,4534,9467
2,92,Unidade,Unidade,28,996
3,84,Cafeteria,Comedoria - Cafeteria,900,947
4,76,Galpão,Galpão,750,784
5,93,Biblioteca,Biblioteca,484,521
6,86,Garimpo,Espaço Lúdico,509,517
7,92,Marquise,Espaços de passagem / Hall / Escada,134,516
8,83,Cafeteria,Comedoria - Cafeteria,439,498
9,79,Quiosque A,Quiosque,345,483


In [ ]:
df_unidades = pd.read_csv(
    'unidades.csv',
    dtype={'uo': str},
).rename(columns={'macro-região': 'regiao', 'unidade': 'nome_unidade'})

df_raw_acoes_preview['uo'] = df_raw_acoes_preview['uo'].astype(str)

df_raw_acoes_preview = df_raw_acoes_preview.merge(
    df_unidades,
    on='uo',
    how='left',
)

# UOs sem correspondencia no CSV
sem_unidade = df_raw_acoes_preview[df_raw_acoes_preview['nome_unidade'].isna()]['uo'].unique()
if len(sem_unidade):
    print(f'UOs sem cadastro em unidades.csv: {sorted(sem_unidade)}')


## Diagnóstico: qualidade do campo `lugares`

`lugares` é um campo de nível de ação (atividade_id), não de sessão. O erro típico de cadastro é
preencher a **capacidade total da ação** (soma de todas as sessões) em vez da capacidade **por sessão**.

A lógica do diagnóstico, por ação:
- conta `n_sessoes` e calcula `lugares_por_sessao = lugares / n_sessoes`
- compara `presentes` (mediana por sessão) com `lugares` e com `lugares_por_sessao`
- classifica cada ação em: **plausível** · **suspeito_total** · **lugares_menor_presentes** · **sem_presentes** · **indefinido**

In [ ]:

# ── Diagnóstico: qualidade do campo 'lugares' ──────────────────────────────
# Trabalha sobre df_raw_acoes_preview (já enriquecido, antes do export).

_diag = df_raw_acoes_preview.copy()
_diag['presentes'] = pd.to_numeric(_diag['presentes'], errors='coerce')
_diag['lugares']   = pd.to_numeric(_diag['lugares'],   errors='coerce')

# Nível de ação: n_sessoes, lugares (é o mesmo em todas as linhas da ação),
# estimativa_publico e mediana de presentes por sessão.
_por_acao = (
    _diag[_diag['lugares'].notna() & (_diag['lugares'] > 0)]
    .groupby('atividade_id')
    .agg(
        n_sessoes            = ('sessao_id',         'nunique'),
        lugares              = ('lugares',            'first'),
        estimativa_publico   = ('estimativa_publico', 'first'),
        presentes_mediana    = ('presentes',          'median'),
        presentes_max        = ('presentes',          'max'),
    )
    .reset_index()
)

_por_acao['lugares_por_sessao'] = _por_acao['lugares'] / _por_acao['n_sessoes']

TOL_PROXIMO  = 0.30   # dentro de 30 % → "próximo"
TOL_LONGE    = 2.0    # mais de 200 % de desvio → "longe"

def _classifica(row):
    p  = row['presentes_mediana']
    l  = row['lugares']
    ls = row['lugares_por_sessao']

    if pd.isna(p) or p <= 0:
        return 'sem_presentes'
    if l < p:
        return 'lugares_menor_presentes'   # impossível: presentes > capacidade

    err_direto   = abs(l  - p) / p
    err_dividido = abs(ls - p) / p

    # Suspeita de erro total: lugares/n_sessoes é bem melhor que lugares
    if err_dividido < TOL_PROXIMO and err_direto > TOL_LONGE:
        return 'suspeito_total'
    if err_direto < TOL_PROXIMO:
        return 'plausivel'
    return 'indefinido'

_por_acao['diagnostico_lugares'] = _por_acao.apply(_classifica, axis=1)

# ── Resumo ─────────────────────────────────────────────────────────────────
print("=== Diagnóstico do campo 'lugares' (nível de ação) ===\n")
_contagens = _por_acao['diagnostico_lugares'].value_counts()
_total = len(_por_acao)
for cat, n in _contagens.items():
    print(f"  {cat:<30} {n:>6,}  ({100*n/_total:.1f} %)")

print(f"\n  Total de ações com 'lugares' preenchido: {_total:,}")

# ── Amostra: suspeitos de erro ──────────────────────────────────────────────
_suspeitos = (
    _por_acao[_por_acao['diagnostico_lugares'] == 'suspeito_total']
    .sort_values('n_sessoes', ascending=False)
)
print(f"\n--- Suspeito de ser total (lugares = capacidade × sessões): {len(_suspeitos):,} ações ---")
display(
    _suspeitos[['atividade_id', 'n_sessoes', 'lugares', 'lugares_por_sessao',
                'presentes_mediana', 'estimativa_publico']]
    .head(20)
    .reset_index(drop=True)
)

# ── Amostra: presentes > lugares ────────────────────────────────────────────
_inconsistentes = (
    _por_acao[_por_acao['diagnostico_lugares'] == 'lugares_menor_presentes']
    .sort_values('presentes_max', ascending=False)
)
print(f"\n--- Presentes > Lugares (inconsistência clara): {len(_inconsistentes):,} ações ---")
display(
    _inconsistentes[['atividade_id', 'n_sessoes', 'lugares', 'lugares_por_sessao',
                      'presentes_mediana', 'presentes_max', 'estimativa_publico']]
    .head(20)
    .reset_index(drop=True)
)


In [ ]:
output_file = OUTPUT_PATH / "apresentacoes_longitudinal.xlsx"
df_raw_acoes_preview.to_excel(output_file, index=False)
print(f"Exportado {len(df_raw_acoes_preview)} linhas para {output_file}")